In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv("X:/AI DataScience projects/Assignment 8/capstone-project-KarunBhandari/Weather_&_Climate_Data/Processed Datas/5yeargrouped.csv")

In [3]:
# 1. Climate Indices
# Heat Stress Index (simple: temperature + humidity combined effect)
df['Heat_Stress_Index'] = df['Temp_2m'] + 0.1 * df['Humidity_2m']

# Drought Index (simple proxy: low precipitation + high temp)
# Scale both components
precip_scaled = (df['Precip'] - df['Precip'].min()) / (df['Precip'].max() - df['Precip'].min())
temp_scaled = (df['Temp_2m'] - df['Temp_2m'].min()) / (df['Temp_2m'].max() - df['Temp_2m'].min())
df['Drought_Index'] = (1 - precip_scaled) + temp_scaled

In [4]:
# 2. Seasonal Indicators
# 1 if Monsoon (June-Sept), else 0
df['Interval_Start'] = pd.to_datetime(df['Interval_Start'])
df['Month'] = df['Interval_Start'].dt.month
df['is_monsoon'] = df['Month'].apply(lambda x: 1 if x in [6,7,8,9] else 0)
# 1 if Winter (Dec-Feb)
df['is_winter'] = df['Month'].apply(lambda x: 1 if x in [12,1,2] else 0)


In [5]:
# 3. Lag Features
# Previous day's temp and precipitation (basic lagged memory)
df['temp_2m_lag1'] = df['Temp_2m'].shift(1)
df['precip_lag1'] = df['Precip'].shift(1)

In [6]:
# 5. Derived Features
df['wetbulb_diff'] = df['Temp_2m'] - df['WetBulbTemp_2m']
df['avg_windspeed'] = (df['WindSpeed_10m'] + df['WindSpeed_50m']) / 2
df['max_avg_windspeed'] = (df['MaxWindSpeed_10m'] + df['MaxWindSpeed_50m']) / 2
#df.to_csv("../Weather_&_Climate_Data/Processed Datas/FE_before_norm.csv")

In [9]:
# 6. Normalization (for modeling)
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[['Temp_2m', 'Pressure', 'Humidity_2m','WetBulbTemp_2m','EarthSkinTemp', 'WindSpeed_10m',
                                         'Heat_Stress_Index']])
scaled_df = pd.DataFrame(scaled_features, 
                         columns=['Temp_2m_scaled', 'Pressure_scaled', 'Humidity_2m_scaled','WetBulbTemp_2m_scaled','EarthSkinTemp_scaled', 'WindSpeed_10m_scaled',
                                         'Heat_Stress_Index_scaled'])
scaled_df.to_csv("Testforscaleddata.csv")

In [20]:

df_dropped = df.drop(['Temp_2m', 'Pressure', 'Humidity_2m', 'WetBulbTemp_2m', 
                      'EarthSkinTemp', 'WindSpeed_10m', 'Heat_Stress_Index'], axis=1)

# Now concatenate with scaled_df
df_scaled_full = pd.concat([df_dropped, scaled_df], axis=1)

#  if scaled columns exist
#print("Final DataFrame Columns:", df_scaled_full.columns)
df_scaled = df_scaled_full[["Latitude","Longitude","Year", "Month", 'Drought_Index', 'temp_2m_lag1',
       'precip_lag1', 'wetbulb_diff', 'avg_windspeed', 'max_avg_windspeed',
       'Temp_2m_scaled', 'Pressure_scaled', 'Humidity_2m_scaled',
       'WetBulbTemp_2m_scaled', 'EarthSkinTemp_scaled', 'WindSpeed_10m_scaled',
       'Heat_Stress_Index_scaled']]
df_scaled.to_csv("../Weather_&_Climate_Data/Processed Datas/Feature_Engineering/climate_df_scaled.csv", index=False)
print("✅ Feature Engineering Complete! New dataset shape:", df.shape)


✅ Feature Engineering Complete! New dataset shape: (620, 33)
